## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip install --force-reinstall --no-cache-dir typing_extensions==4.11.0
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
## Sometimes needed in runpod to make sure it goes to the network volumne
import os

os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HOME"] = "/workspace/hf_home"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor, CLIPVisionModel
from datasets import load_from_disk, concatenate_datasets, load_dataset
import pandas as pd
import numpy as np
import os
from collections import defaultdict
import copy
from typing import Optional

## Attribute Configurations

In [ ]:
# Custom Datasets Configuration
dataset = load_dataset("")
num_classes = 0
dataset_name = ""
size_nums = [] 
fine_tuned_model_name = "tanganke/clip-vit-base-patch32_mnist"

# Regular for CLIP
domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
transformation = "Standard" # "Standard" 
model_name = "CLIP"
refer = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
indices = [i for i in range(refer.vision_model.encoder.config.num_hidden_layers)]
folder = f"./Results/{model_name}/{dataset_name}_{domain}/Entire_Transformation_Matrix_W"
device = "cuda" if torch.cuda.is_available() else "cpu"

## Classes Preparation

In [ ]:
class Hooks(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.CLS = []
    
    def forward(self, images):
        self.CLS = []
        
        hidden_states = self.model.vision_model.embeddings(images)
        hidden_states = self.model.vision_model.pre_layrnorm(hidden_states)

        for encoder_layer in self.model.vision_model.encoder.layers:
            hidden_states = encoder_layer(hidden_states, attention_mask=None, causal_attention_mask=None, output_attentions=False)[0]
            self.CLS.append(hidden_states[:, 0, :])

        return self.CLS

In [ ]:
class Augmented(torch.nn.Module):
    def __init__(self, model, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage
    
    @staticmethod
    def _get_vector_norm(tensor: torch.Tensor) -> torch.Tensor:
        square_tensor = torch.pow(tensor, 2)
        sum_tensor = torch.sum(square_tensor, dim=-1, keepdim=True)
        normed_tensor = torch.pow(sum_tensor, 0.5)
        return normed_tensor
    
    def forward(self, pixel_values, input_ids, attention_mask):
        # Vision Side
        hidden_states = self.model.vision_model.embeddings(pixel_values)
        hidden_states = self.model.vision_model.pre_layrnorm(hidden_states)

        for i, encoder_layer in enumerate(self.model.vision_model.encoder.layers):
            hidden_states = encoder_layer(hidden_states, attention_mask=None, causal_attention_mask=None, output_attentions=False)[0]
            if i == self.transform_stage:
                if self.W is None:
                    self.W = torch.eye(hidden_states.shape[-1], device=hidden_states.device, dtype=hidden_states.dtype)
                hidden_states[:, 0, :] = hidden_states[:, 0, :] @ self.W
                break
        
        pooled_output = self.model.vision_model.post_layernorm(hidden_states[:, 0, :])
        image_embeds = self.model.visual_projection(pooled_output)

        # Text Side
        text_outputs = self.model.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embeds = text_outputs.pooler_output
        text_embeds = self.model.text_projection(text_embeds)

        # L2 Norm
        image_embeds = image_embeds / self._get_vector_norm(image_embeds)
        text_embeds = text_embeds / self._get_vector_norm(text_embeds)

        # Cosine Similarity as Logits
        logits_per_text = torch.matmul(text_embeds, image_embeds.t().to(text_embeds.device))
        logits_per_text = logits_per_text * self.model.logit_scale.exp().to(text_embeds.device)

        logits_per_image = logits_per_text.t()

        return logits_per_image, pooled_output # logits, raw vision cls token

In [ ]:
def cosineSimilarity(fine_tuned_cls, aug_cls):
    eps = 1e-8
    out_aug = F.normalize(aug_cls, dim=1, eps=eps)
    out_fine = F.normalize(fine_tuned_cls, dim=1, eps=eps)

    cos_sim = (out_aug * out_fine).sum(dim=1).mean().item()
    return cos_sim

## Loading Models

In [ ]:
f_t_vision_model = CLIPVisionModel.from_pretrained(f'{fine_tuned_model_name}')
fine_tuned = copy.deepcopy(refer)
fine_tuned.vision_model.load_state_dict(f_t_vision_model.vision_model.state_dict())
fine_tuned = Augmented(fine_tuned)
fine_tuned = fine_tuned.eval().to(device)

base_H = Hooks(copy.deepcopy(refer)).to(device)
fine_tuned_H = Hooks(copy.deepcopy(fine_tuned.model)).to(device)

base = Augmented(copy.deepcopy(refer))
base = base.eval().to(device)

## Dataset Preparation

In [ ]:
train = load_from_disk(f'/workspace/preprocessed/{dataset_name}/train_processed')
val = load_from_disk(f'/workspace/preprocessed/{dataset_name}/val_processed')
test = load_from_disk(f'/workspace/preprocessed/{dataset_name}/test_processed')

train.set_format(type='torch', columns=["image", "label", "pixel_values"])
val.set_format(type='torch', columns=["image", "label", "pixel_values"])
test.set_format(type='torch', columns=["image", "label", "pixel_values"])

full_train = concatenate_datasets([train, val])
full_train_size = len(full_train)

In [ ]:
train.set_format(type='torch', columns=["image", "label", "pixel_values"])
val.set_format(type='torch', columns=["image", "label", "pixel_values"])
test.set_format(type='torch', columns=["image", "label", "pixel_values"])

def collate_fn(batch):
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images,
        "labels": labels
    }

full_train_loader = DataLoader(full_train, batch_size=64, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

In [ ]:
# Full processor for Image and Text
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
label_names = dataset["train"].features["label"].names

label_texts = [f"a photo of a {label_names[i]}" for i in range(num_classes)]
text_inputs = processor(text=label_texts, return_tensors="pt", padding=True, truncation=True)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

## Dataset Prep

In [ ]:
train_size = 0

labels = train["label"]

label_to_indices = defaultdict(list)

for idx, label in enumerate(labels):
    label = int(label)
    label_to_indices[label].append(idx)

filtered_train = {
    label: train.select(indices) for label, indices in label_to_indices.items()
}

for label, ds in filtered_train.items():
    print(f"Label {label}: {len(ds)} examples")

In [ ]:
def extract_vectors(train_loader):
    Z0 = {i: [] for i in indices}
    Z1 = []

    with torch.no_grad():
        for batch in tqdm(train_loader, desc="Extracting"):
            images = batch["pixel_values"].to(device, non_blocking=True)

            if domain == "Fine_Tuned_Layer_Skipping":
                out_fine_tuned = fine_tuned_H(images)
                
                for i in indices:
                    Z0[i].append(out_fine_tuned[i].float().cpu())
                Z1.append(out_fine_tuned[-1].float().cpu())
            else:
                out_base = base_H(images)
                out_fine_tuned = fine_tuned_H(images)

                for i in indices:
                    Z0[i].append(out_base[i].float().cpu())
                Z1.append(out_fine_tuned[-1].float().cpu())
                
    Z1_last = torch.cat(Z1)
    Z1 = Z1_last.cpu().numpy()

    W = {}
    resid = {}

    for i, val in Z0.items():
        val = torch.cat(val)
        val = val.cpu().numpy()
        W[i], resid[i], _, _ = np.linalg.lstsq(val, Z1, rcond=None)
    
    return W, resid

In [ ]:
def augment_models(transform_type, W=None):
    if W is None:
        W = {i: None for i in indices}
    aug = {}
    for i in indices:
        reference = copy.deepcopy(refer)
        if domain == "Fine_Tuned_Layer_Skipping":
            reference = copy.deepcopy(fine_tuned.model)
        if transform_type == "Standard":
            model = Augmented(copy.deepcopy(reference), W=W[i], transform_stage=i)
        model = model.eval().to(device)
        aug[i] = model
    return aug

In [ ]:
def evaluate(aug):
    correct_base = 0
    correct_fine_tuned = 0
    total_samples = 0

    correct = {i: 0 for i in indices}
    co_sim_cls = {i: [] for i in indices}

    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        total_samples += labels.size(0)

        with torch.no_grad():
            # Base Model
            logits_base, cls_base = base(images, **text_inputs)
            predicted = logits_base.argmax(dim=1)
            correct_base += (predicted == labels).sum().item()

            # Fine-Tuned Model
            logits_fine_tuned, cls_fine_tuned = fine_tuned(images, **text_inputs)
            predicted = logits_fine_tuned.argmax(dim=1)
            correct_fine_tuned += (predicted == labels).sum().item()

            # Augmented Models
            for i in indices:
                logits_aug, cls_aug = aug[i](images, **text_inputs)
                predicted = logits_aug.argmax(dim=1)
                correct[i] += (predicted == labels).sum().item()
                co_sim_cls[i].append(cosineSimilarity(cls_fine_tuned, cls_aug))

    base_acc = correct_base / total_samples
    fine_tuned_acc = correct_fine_tuned / total_samples

    for i in indices:
        correct[i] = correct[i] / total_samples
        co_sim_cls[i] = np.mean(co_sim_cls[i])

    print(f"Augmented {model_name} on {dataset_name} Results")
    for i in indices:
        print(f"\tAugmented {i} - Last ({indices[-1]}) Layer Accuracy: {correct[i]}")
        print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i} Layer: {co_sim_cls[i]:.4f}")
    print(f"Base Accuracy: {base_acc:.4f}")
    print(f"Fine-Tuned Accuracy: {fine_tuned_acc:.4f}")

    return base_acc, fine_tuned_acc, correct, co_sim_cls

In [ ]:
def save_results(correct, co_sim_cls, transformation, train_size=None, save_W=False, W=None, resid=None):
    name = f"{transformation}_"
    folder = f"./Results/{dataset_name}/{domain}/Entire_Transformation_Matrix_W"
    os.makedirs(folder, exist_ok=True)

    data = {
        'Classification_Accuracy': [correct[i] for i in indices],
        'CLS_Cosine_Similarity': [co_sim_cls[i] for i in indices]
    }

    if transformation == "Standard":
        data["Train_Data_Size"] = [train_size] * len(indices),
        data["Residuals"] = [resid[i] for i in indices],
        name += f"{train_size}_"
    if save_W:
        data["W"] = [W[i] for i in indices]

    df = pd.DataFrame(data, index=indices)
    name += "Results.json"
    path = os.path.join(folder, name)
    df.to_json(path, orient="records", indent=2)

## Evaluating

In [ ]:
print(f"{model_name} - {dataset_name}: {domain}\n")

for s in size_nums:
    image_per_label = s

    sorted = []
    for i in range(num_classes):
        num_indices = min(image_per_label, len(filtered_train[i]))
        sorted.append(filtered_train[i].select(range(num_indices)))
    
    train_dataset = concatenate_datasets(sorted)
    train_size= len(train_dataset)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    print(f"Results for {model_name} on {dataset_name}: {train_size} Training Images ({image_per_label} images/label)")

    W, resid = extract_vectors(train_loader)
    aug = augment_models(transformation, W=W)
    base_acc, fine_tuned_acc, correct, co_sim_cls = evaluate(aug)
    save_results(correct, co_sim_cls, transformation, train_size=train_size, resid=resid)

print(f"Results for {model_name} on {dataset_name}: {full_train_size} Training Images (All)")
W, resid = extract_vectors(full_train_loader)
aug = augment_models(transformation, W=W)
base_acc, fine_tuned_acc, correct, co_sim_cls = evaluate(aug)
save_results(correct, co_sim_cls, transformation, train_size=full_train_size, resid=resid)